# The Human Cost of Civilian Targeting
## 01 — Data cleaning and quality audit

**Central question:** *How has violence against civilians changed globally between 2021 and 2025, and where has its human cost been most severe?*

This notebook prepares the ACLED data used by the four research questions and by the final Tableau dashboard. It keeps only the variables required for the regional comparisons, the event-form analysis, the Relative Attention Index, and the two country event maps.


### Analytical scope and limitations

- `civilian_targeting` identifies events in which civilians are recorded as the main or sole target; it does not capture every indirect consequence of conflict.
- `fatalities` is ACLED's estimate of deaths reported for the event as a whole. It is therefore described as **reported fatalities in civilian-targeting events**, not as an exact count of civilian deaths.
- The data cover 2021–2024 in full and 2025 only through **25 August**. The 2025 column is always marked as partial.
- Event frequency, total reported fatalities, and fatalities per event measure different dimensions and are kept separate.


## 1. Setup


In [1]:
from pathlib import Path
import platform

import numpy as np
import pandas as pd
import pyarrow

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"pyarrow: {pyarrow.__version__}")


Python: 3.13.7
pandas: 3.0.5
pyarrow: 25.0.1


In [2]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_DIR = next(
    (path for path in [CURRENT_DIR, *CURRENT_DIR.parents]
     if (path / "requirements.txt").exists() and (path / "data").is_dir()),
    CURRENT_DIR,
)
START_YEAR = 2021
END_YEAR = 2025
CHUNK_SIZE = 200_000

csv_candidates = [
    path for path in PROJECT_DIR.glob("*acled*.csv*")
    if path.is_file() and "clean" not in path.name.lower()
]
if not csv_candidates:
    raise FileNotFoundError("No ACLED CSV found in the project directory.")

RAW_CSV = max(csv_candidates, key=lambda path: path.stat().st_size)
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_PARQUET = OUTPUT_DIR / "acled_civilian_targeting_2021_2025.parquet"
AUDIT_CSV = OUTPUT_DIR / "cleaning_audit.csv"
COVERAGE_CSV = OUTPUT_DIR / "year_coverage.csv"

print(f"Input: {RAW_CSV.name} ({RAW_CSV.stat().st_size / 1024**3:.2f} GB)")
print(f"Output: {CLEAN_PARQUET.relative_to(PROJECT_DIR)}")


Input: acled_global_political_violence_2020-01-01_2026-07-31_download-2026-08-25.csv.csv (1.03 GB)
Output: data/processed/acled_civilian_targeting_2021_2025.parquet


## 2. Variables used in the project

Only twelve source fields are required. `timestamp` is used exclusively to retain the most recent version of a duplicated event and is removed before export.


In [3]:
USECOLS = [
    "event_id_cnty", "event_date", "year", "event_type", "sub_event_type",
    "civilian_targeting", "region", "country", "latitude", "longitude",
    "fatalities", "timestamp",
]

preview = pd.read_csv(RAW_CSV, nrows=5, usecols=USECOLS, dtype="string")
display(preview)
print(f"Selected variables: {len(USECOLS)}")


,event_id_cnty,event_date,year,event_type,sub_event_type,civilian_targeting,region,country,latitude,longitude,fatalities,timestamp
0,TUR10582,2020-01-01,2020,Strategic developments,Arrests,<NA>,Middle East,Turkey,41.6757,26.5587,0,1578503874
1,TUR10583,2020-01-01,2020,Strategic developments,Arrests,<NA>,Middle East,Turkey,37.8582,27.2607,0,1578503874
2,IRN5946,2020-01-01,2020,Protests,Peaceful protest,<NA>,Middle East,Iran,29.6103,52.5311,0,1578503875
3,IRN5874,2020-01-01,2020,Protests,Peaceful protest,<NA>,Middle East,Iran,36.3156,59.5680,0,1578503875
4,TUN6016,2020-01-01,2020,Protests,Peaceful protest,<NA>,Northern Africa,Tunisia,32.9297,10.4518,0,1578512391


Selected variables: 12


## 3. Chunked loading and project filter


In [4]:
filtered_chunks = []
load_audit_rows = []

reader = pd.read_csv(
    RAW_CSV,
    usecols=USECOLS,
    dtype="string",
    chunksize=CHUNK_SIZE,
    encoding="utf-8-sig",
)

for chunk_number, chunk in enumerate(reader, start=1):
    raw_year = pd.to_numeric(chunk["year"], errors="coerce")
    in_period = raw_year.between(START_YEAR, END_YEAR, inclusive="both")
    is_civilian_targeting = chunk["civilian_targeting"].str.strip().eq("Civilian targeting")
    keep = in_period.fillna(False) & is_civilian_targeting.fillna(False)

    filtered_chunks.append(chunk.loc[keep].copy())
    load_audit_rows.append({
        "chunk": chunk_number,
        "rows_read": len(chunk),
        "rows_in_period": int(in_period.sum()),
        "rows_kept": int(keep.sum()),
    })

df = pd.concat(filtered_chunks, ignore_index=True)
load_audit = pd.DataFrame(load_audit_rows)

display(load_audit[["rows_read", "rows_in_period", "rows_kept"]].sum().to_frame("total"))
print(f"Filtered rows loaded: {len(df):,}")


,total
rows_read,1914454
rows_in_period,1644002
rows_kept,251579


Filtered rows loaded: 251,579


## 4. Type normalization and validity rules


In [5]:
text_columns = [
    "event_id_cnty", "event_type", "sub_event_type",
    "civilian_targeting", "region", "country",
]
for column in text_columns:
    df[column] = df[column].str.strip().replace("", pd.NA)

df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df["year_reported"] = pd.to_numeric(df["year"], errors="coerce").astype("Int16")
df["year"] = df["event_date"].dt.year.astype("Int16")
df["year_mismatch"] = df["year"].ne(df["year_reported"]).fillna(True)

df["fatalities"] = pd.to_numeric(df["fatalities"], errors="coerce").astype("Int64")
df["fatalities_invalid"] = df["fatalities"].lt(0).fillna(False)
df.loc[df["fatalities_invalid"], "fatalities"] = pd.NA

for column in ["latitude", "longitude"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["coordinates_valid"] = (
    df["latitude"].between(-90, 90)
    & df["longitude"].between(-180, 180)
).fillna(False)
df.loc[~df["coordinates_valid"], ["latitude", "longitude"]] = np.nan

df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce").astype("Int64")
df.info(memory_usage="deep")


<class 'pandas.DataFrame'>
RangeIndex: 251579 entries, 0 to 251578
Data columns (total 16 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   event_id_cnty       251579 non-null  string        
 1   event_date          251579 non-null  datetime64[us]
 2   year                251579 non-null  Int16         
 3   event_type          251579 non-null  string        
 4   sub_event_type      251579 non-null  string        
 5   civilian_targeting  251579 non-null  string        
 6   region              251579 non-null  string        
 7   country             251579 non-null  string        
 8   latitude            251579 non-null  Float64       
 9   longitude           251579 non-null  Float64       
 10  fatalities          251579 non-null  Int64         
 11  timestamp           251579 non-null  Int64         
 12  year_reported       251579 non-null  Int16         
 13  year_mismatch       251579 non-null  boo

## 5. Duplicates and unusable records

Duplicate IDs are resolved by retaining the most recent ACLED timestamp. Records missing a field required by the four analyses are excluded. Invalid coordinates do not remove an event; they only prevent it from appearing on the Tableau point map.


In [6]:
rows_before_rules = len(df)
duplicate_rows = int(df.duplicated(subset="event_id_cnty", keep=False).sum())

df = (
    df.sort_values(["event_id_cnty", "timestamp"], na_position="first")
      .drop_duplicates(subset="event_id_cnty", keep="last")
)

required_fields = [
    "event_id_cnty", "event_date", "event_type", "sub_event_type",
    "civilian_targeting", "region", "country", "fatalities",
]
valid_core = (
    df[required_fields].notna().all(axis=1)
    & df["year"].between(START_YEAR, END_YEAR, inclusive="both")
    & ~df["year_mismatch"]
)
rows_invalid_core = int((~valid_core).sum())
df = df.loc[valid_core].copy()

print(f"Rows before validity rules: {rows_before_rules:,}")
print(f"Rows involved in duplicate IDs: {duplicate_rows:,}")
print(f"Rows removed for missing/invalid required fields: {rows_invalid_core:,}")
print(f"Clean rows: {len(df):,}")


Rows before validity rules: 251,579
Rows involved in duplicate IDs: 0
Rows removed for missing/invalid required fields: 0
Clean rows: 251,579


## 6. Coverage and quality audit


In [7]:
year_coverage = (
    df.groupby("year", observed=True)
      .agg(
          first_date=("event_date", "min"),
          last_date=("event_date", "max"),
          events=("event_id_cnty", "size"),
          fatalities=("fatalities", "sum"),
      )
      .reset_index()
)
year_coverage["is_partial_year"] = year_coverage["last_date"].dt.strftime("%m-%d").ne("12-31")

quality_audit = pd.DataFrame({
    "check": [
        "rows_clean", "unique_event_ids", "duplicate_rows",
        "missing_event_date", "year_mismatch", "missing_fatalities",
        "negative_fatalities_found", "invalid_or_missing_coordinates",
        "first_event_date", "last_event_date",
    ],
    "value": [
        len(df), df["event_id_cnty"].nunique(), duplicate_rows,
        int(df["event_date"].isna().sum()), int(df["year_mismatch"].sum()),
        int(df["fatalities"].isna().sum()), int(df["fatalities_invalid"].sum()),
        int((~df["coordinates_valid"]).sum()),
        df["event_date"].min().date().isoformat(),
        df["event_date"].max().date().isoformat(),
    ],
})

display(year_coverage)
display(quality_audit)

assert df["event_id_cnty"].is_unique
assert df["event_date"].notna().all()
assert df["year"].between(START_YEAR, END_YEAR).all()
assert df["civilian_targeting"].eq("Civilian targeting").all()
assert df["fatalities"].ge(0).all()
assert df.loc[df["coordinates_valid"], "latitude"].between(-90, 90).all()
assert df.loc[df["coordinates_valid"], "longitude"].between(-180, 180).all()
assert df["event_date"].min() == pd.Timestamp("2021-01-01")
assert df["event_date"].max() == pd.Timestamp("2025-08-25")
print("✓ All integrity checks passed.")


,year,first_date,last_date,events,fatalities,is_partial_year
0,2021,2021-01-01,2021-12-31,41510,44645,False
1,2022,2022-01-01,2022-12-31,51356,55704,False
2,2023,2023-01-01,2023-12-31,54735,71692,False
3,2024,2024-01-01,2024-12-31,60181,78960,False
4,2025,2025-01-01,2025-08-25,43797,52967,True


,check,value
0,rows_clean,251579
1,unique_event_ids,251579
2,duplicate_rows,0
3,missing_event_date,0
4,year_mismatch,0
5,missing_fatalities,0
6,negative_fatalities_found,0
7,invalid_or_missing_coordinates,0
8,first_event_date,2021-01-01
9,last_event_date,2025-08-25


✓ All integrity checks passed.


## 7. Final project dataset


In [8]:
ANALYSIS_COLUMNS = [
    "event_id_cnty", "event_date", "year", "event_type", "sub_event_type",
    "civilian_targeting", "region", "country", "latitude", "longitude",
    "fatalities",
]
df = df[ANALYSIS_COLUMNS].copy()

for column in ["event_type", "sub_event_type", "civilian_targeting", "region", "country"]:
    df[column] = df[column].astype("category")

df = df.sort_values(["event_date", "event_id_cnty"]).reset_index(drop=True)
df.to_parquet(CLEAN_PARQUET, index=False, compression="zstd")
quality_audit.to_csv(AUDIT_CSV, index=False)
year_coverage.to_csv(COVERAGE_CSV, index=False)

print(f"Dataset: {CLEAN_PARQUET} ({CLEAN_PARQUET.stat().st_size / 1024**2:.1f} MB)")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]}")
print(f"Audit: {AUDIT_CSV}")
print(f"Coverage: {COVERAGE_CSV}")
display(df.head())


Dataset: /Users/nicole/Desktop/projects/datavis/data/processed/acled_civilian_targeting_2021_2025.parquet (3.0 MB)
Rows × columns: 251,579 × 11
Audit: /Users/nicole/Desktop/projects/datavis/data/processed/cleaning_audit.csv
Coverage: /Users/nicole/Desktop/projects/datavis/data/processed/year_coverage.csv


,event_id_cnty,event_date,year,event_type,sub_event_type,civilian_targeting,region,country,latitude,longitude,fatalities
0,AFG50076,2021-01-01,2021,Violence against civilians,Attack,Civilian targeting,Caucasus and Central Asia,Afghanistan,34.5195,65.2509,1
1,AFG50185,2021-01-01,2021,Violence against civilians,Attack,Civilian targeting,Caucasus and Central Asia,Afghanistan,31.6133,65.7101,1
2,AFG59611,2021-01-01,2021,Violence against civilians,Attack,Civilian targeting,Caucasus and Central Asia,Afghanistan,36.905,66.1834,1
3,AZE17124,2021-01-01,2021,Explosions/Remote violence,Remote explosive/landmine/IED,Civilian targeting,Caucasus and Central Asia,Azerbaijan,39.5982,47.1469,0
4,BGD18666,2021-01-01,2021,Violence against civilians,Attack,Civilian targeting,South Asia,Bangladesh,20.8583,92.2977,1
